# ISLES'24 NCCT baseline preparation — fixed for Zenodo v7 paths

Run **Runtime → Run all**.

This version is based on the actual archive paths observed in the persistent log: subjects such as `sub-stroke0001`, acute session `ses-01`, and follow-up session `ses-02`.

It extracts only acute NCCT and the final-infarct mask for the first predeclared baseline. Progress and errors are persisted in Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, tempfile

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
ARCHIVE = ROOT / 'raw' / 'train.7z'
SOURCE = ROOT / 'ncct_source'
RESULTS = ROOT / 'results'
NNUNET_RAW = ROOT / 'nnUNet_raw'
LOG = RESULTS / 'ncct_prepare_fixed.log'
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')

RESULTS.mkdir(parents=True, exist_ok=True)
NNUNET_RAW.mkdir(parents=True, exist_ok=True)

def log(msg):
    text = str(msg)
    print(text, flush=True)
    with LOG.open('a', encoding='utf-8') as handle:
        handle.write(text + '\n')

def run_logged(cmd):
    cmd = [str(x) for x in cmd]
    log('$ ' + ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with LOG.open('a', encoding='utf-8') as handle:
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            handle.write(line)
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)

LOG.write_text('ISLES24 NCCT fixed preparation started\n', encoding='utf-8')
if not ARCHIVE.is_file():
    raise FileNotFoundError(ARCHIVE)
log(f'Archive size: {ARCHIVE.stat().st_size} bytes')


## Install tools and load the latest repository snapshot


In [ ]:
run_logged(['apt-get', 'update', '-qq'])
run_logged(['apt-get', 'install', '-y', '-qq', 'p7zip-full', 'unzip'])
if REPO.exists():
    shutil.rmtree(REPO)
if REPO_ZIP.exists():
    REPO_ZIP.unlink()
run_logged([
    'curl', '-fL', '--retry', '20', '--retry-delay', '3', '--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o', REPO_ZIP,
])
run_logged(['unzip', '-q', REPO_ZIP, '-d', '/content'])
os.chdir(REPO)
run_logged(['python', '-m', 'pip', 'install', '-q', '-e', '.'])
log(f'Repository ready: {REPO}')


## Validate path matching against the actual Zenodo v7 naming


In [ ]:
NCCT_RE = re.compile(r'/(?:rawdata|raw_data)/(sub-[^/]+)/ses-0*1/[^/]*_ncct\.nii\.gz$', re.I)
MASK_RE = re.compile(r'/derivatives/(sub-[^/]+)/ses-0*2/[^/]*_lesion-msk\.nii\.gz$', re.I)

sample_ncct = '/train/rawdata/sub-stroke0001/ses-01/sub-stroke0001_ses-01_ncct.nii.gz'
sample_mask = '/train/derivatives/sub-stroke0001/ses-02/sub-stroke0001_ses-02_space-ncct_lesion-msk.nii.gz'
assert NCCT_RE.search(sample_ncct), sample_ncct
assert MASK_RE.search(sample_mask), sample_mask
log('Path-pattern self-test passed for ses-01 / ses-02.')


## Inspect the archive and extract 149 paired NCCT/mask cases

Extraction is resumable in 10-subject batches.


In [ ]:
seven = shutil.which('7zz') or shutil.which('7z')
if seven is None:
    raise RuntimeError('7-Zip not found')

log('Listing archive members. This can take several minutes on Drive...')
listing = subprocess.run(
    [seven, 'l', '-slt', str(ARCHIVE)],
    capture_output=True,
    text=True,
    errors='replace',
    check=True,
)
members = [
    line[7:].strip().replace('\\', '/')
    for line in listing.stdout.splitlines()
    if line.startswith('Path = ')
]
log(f'Archive entries reported by 7-Zip: {len(members)}')

ncct = {}
masks = {}
for member in members:
    path = '/' + member.lstrip('./')
    match = NCCT_RE.search(path)
    if match:
        sid = match.group(1).lower()
        if sid in ncct:
            raise RuntimeError(f'Duplicate NCCT for {sid}')
        ncct[sid] = member
        continue
    match = MASK_RE.search(path)
    if match:
        sid = match.group(1).lower()
        if sid in masks:
            raise RuntimeError(f'Duplicate lesion mask for {sid}')
        masks[sid] = member

shared = sorted(set(ncct) & set(masks))
log(f'NCCT found: {len(ncct)}; masks found: {len(masks)}; paired: {len(shared)}')
if len(ncct) != 149 or len(masks) != 149 or len(shared) != 149:
    log('Diagnostic examples from the archive:')
    for member in [m for m in members if 'ncct' in m.lower() or 'lesion' in m.lower()][:120]:
        log('  ' + member)
    raise RuntimeError(
        f'Expected 149 paired cases, got NCCT={len(ncct)}, masks={len(masks)}, paired={len(shared)}. '
        f'Persistent log: {LOG}'
    )

state = SOURCE / '.extract_state'
state.mkdir(parents=True, exist_ok=True)
batches = [shared[i:i+10] for i in range(0, len(shared), 10)]

for batch_index, subjects in enumerate(batches):
    marker = state / f'batch_{batch_index:03d}.done'
    if marker.exists():
        log(f'[{batch_index + 1}/{len(batches)}] skipping completed batch')
        continue

    selected = []
    for sid in subjects:
        selected.extend([ncct[sid], masks[sid]])

    with tempfile.NamedTemporaryFile('w', encoding='utf-8', suffix='.txt', delete=False) as handle:
        listfile = Path(handle.name)
        handle.write('\n'.join(selected) + '\n')

    try:
        run_logged([
            seven, 'x', ARCHIVE, f'-o{SOURCE}', '-aoa', '-scsUTF-8', f'-i@{listfile}'
        ])
    finally:
        listfile.unlink(missing_ok=True)

    marker.write_text('\n'.join(subjects) + '\n', encoding='utf-8')
    log(f'[{batch_index + 1}/{len(batches)}] completed {len(subjects)} subjects')

log('NCCT/mask extraction complete.')


## Audit the extracted NCCT subset


In [ ]:
AUDIT = RESULTS / 'isles2024_ncct_audit.json'
run_logged([
    'python', 'scripts/audit_isles2024.py', SOURCE,
    '--channels', 'ncct',
    '--json', AUDIT,
])
log(f'Audit saved: {AUDIT}')


## Stage Dataset504 for nnU-Net v2


In [ ]:
run_logged([
    'python', 'scripts/stage_isles2024_nnunet.py', SOURCE, NNUNET_RAW,
    '--dataset-id', '504',
    '--dataset-name', 'ISLES2024_NCCT',
    '--channels', 'ncct',
    '--mode', 'copy',
    '--resume',
    '--split-seed', '2026',
])

dataset = NNUNET_RAW / 'Dataset504_ISLES2024_NCCT'
image_count = len(list((dataset / 'imagesTr').glob('*.nii.gz')))
label_count = len(list((dataset / 'labelsTr').glob('*.nii.gz')))
log(f'imagesTr: {image_count}')
log(f'labelsTr: {label_count}')
log(f'dataset.json exists: {(dataset / "dataset.json").exists()}')
log(f'splits_final.json exists: {(dataset / "splits_final.json").exists()}')
if image_count != 149 or label_count != 149:
    raise RuntimeError(f'Expected 149 staged images and labels, got {image_count}/{label_count}')
log('NCCT BASELINE PREPARATION COMPLETE')


## Recovery rule

If Colab disconnects, reopen this notebook and run all again. The extraction batches and staged files are persistent. The diagnostic log is `MyDrive/HybridStrokeSeg/ISLES2024/results/ncct_prepare_fixed.log`.
